# 🔐 PhishingGuard — Entrenamiento optimizado (Google Colab)

**Objetivo:** mejorar *precision* y *accuracy* sin utilizar el conjunto de prueba para ajustar modelos. Se conservan **1 = phishing** y **0 = no phishing**, y la división **60 % entrenamiento, 20 % validación y 20 % prueba**.

**Datos en Drive:** `/content/drive/MyDrive/phishing/Spaphish dataset - DiB.csv` y `legitphish.csv`.

**Mejoras frente al notebook anterior:**
1. Comprobación y separación de correos duplicados exactos para limitar fugas de datos (puede reducir algo el total utilizable).
2. Texto: comparación de TF-IDF por palabras frente a **palabras + n-gramas de caracteres** y distintos grados de regularización.
3. Elección de **umbral** y de **peso texto/URL** usando **solo validación**, sin escoger el mejor resultado del test.
4. Comparación con línea base original y generación de reporte, matrices y modelos exportables.

**Advertencia:** un AUC alto (como 0.987) no garantiza un aumento de precision ni accuracy. Subir el umbral puede mejorar precision, pero también reducir recall. Para phishing se exige en validación un recall mínimo objetivo de 0.95 cuando es alcanzable. No se visitan URLs, solo se procesan sus cadenas.


## 1. Configuración y Drive

In [ ]:
# 1. Conectar Google Drive y configurar los archivos REALES
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/phishing'
EMAIL_PATH = f'{BASE}/Spaphish dataset - DiB.csv'  # nombre tal como aparece en tu Drive
URL_PATH = f'{BASE}/legitphish.csv'

print('Carpeta:', BASE)
if not Path(BASE).is_dir():
    raise FileNotFoundError(f'No se encontró la carpeta {BASE}')
print('Archivos:', sorted(p.name for p in Path(BASE).iterdir() if p.is_file()))

for nombre, ruta in [('SpaPhish', EMAIL_PATH), ('LegitPhish', URL_PATH)]:
    if not Path(ruta).is_file():
        raise FileNotFoundError(f'No se encontró {nombre}: {ruta} (revisa mayúsculas y espacios)')
    print(f'{nombre}: OK → {Path(ruta).name}')

os.makedirs(f'{BASE}/models', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)


## 2. Funciones compartidas (se reutilizan en la app)

In [ ]:
%%writefile /content/features.py
"""Funciones compartidas entre Colab (entrenamiento) y Streamlit (app)."""
import re
import math
import unicodedata
from collections import Counter
from urllib.parse import urlparse
import pandas as pd

URL_REGEX = re.compile(r"(?:https?://|www\.)[^\s<>\"'\)\]]+", re.IGNORECASE)

ACORTADORES = {"bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd",
               "buff.ly", "cutt.ly", "rb.gy", "shorturl.at"}
TLD_SOSPECHOSOS = {"xyz", "top", "club", "click", "link", "info", "online", "site",
                   "tk", "ml", "ga", "cf", "gq", "work", "support", "live", "icu", "buzz"}
PALABRAS_URL = ["login", "signin", "verify", "verifica", "secure", "seguro", "account",
                "cuenta", "update", "actualiza", "bank", "banco", "password", "clave",
                "confirm", "pago", "pay", "wallet", "suspend", "bloque", "recover", "acceso"]

EXT_RIESGOSAS = {".exe", ".scr", ".js", ".vbs", ".bat", ".cmd", ".lnk", ".iso", ".jar",
                 ".zip", ".rar", ".7z", ".html", ".htm", ".docm", ".xlsm", ".pptm", ".msi"}

INDICADORES_TEXTO = {
    "Lenguaje de urgencia": ["urgente", "inmediat", "ultimo aviso", "24 horas", "48 horas",
                             "hoy mismo", "de inmediato", "expira", "vence", "cuanto antes",
                             "ultima oportunidad", "lo antes posible"],
    "Solicitud de credenciales / datos sensibles": ["contrasena", "clave", "usuario", "password",
                             "verifica tu cuenta", "confirma tus datos", "datos bancarios",
                             "tarjeta", "cvv", "numero de cuenta", "iniciar sesion",
                             "actualiza tus datos", "valida tu"],
    "Amenaza o consecuencia negativa": ["suspendida", "bloqueada", "cancelada", "cerrada",
                             "sancion", "multa", "penalidad", "acceso restringido",
                             "desactivada", "suspension"],
    "Lenguaje de autoridad": ["banco", "soporte tecnico", "departamento de", "administrador",
                             "sunat", "gerencia", "recursos humanos", "seguridad informatica",
                             "equipo de seguridad"],
    "Gancho de premio / beneficio": ["ganado", "premio", "gratis", "sorteo", "reembolso",
                             "herencia", "felicidades", "beneficio"],
    "Llamado a hacer clic / abrir adjunto": ["haz clic", "haga clic", "clic aqui", "click",
                             "pulsa", "descarga el adjunto", "abre el archivo", "ingresa aqui"],
}


def _norm(texto):
    texto = unicodedata.normalize("NFD", str(texto).lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")


def extract_urls(texto):
    urls = [u.rstrip(".,;:") for u in URL_REGEX.findall(str(texto or ""))]
    return list(dict.fromkeys(urls))


def _entropia(s):
    if not s:
        return 0.0
    n = len(s)
    return -sum((c / n) * math.log2(c / n) for c in Counter(s).values())


def url_features(url):
    u = str(url).strip()
    con_esquema = bool(re.match(r"^[a-z]+://", u, re.I))
    p = urlparse(u if con_esquema else "http://" + u)
    host = (p.hostname or "").lower()
    path = p.path or ""
    partes = [x for x in host.split(".") if x]
    es_ip = bool(re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}", host))
    tld = partes[-1] if partes else ""
    low = u.lower()
    try:
        tiene_puerto = int(p.port is not None)
    except ValueError:
        tiene_puerto = 1
    n = max(len(u), 1)
    n_dig = sum(c.isdigit() for c in u)
    return {
        "longitud": len(u),
        "longitud_dominio": len(host),
        "longitud_path": len(path),
        "n_puntos": u.count("."),
        "n_subdominios": 0 if es_ip else max(len(partes) - 2, 0),
        "n_guiones_dominio": host.count("-"),
        "n_digitos": n_dig,
        "ratio_digitos": n_dig / n,
        "n_especiales": sum(not c.isalnum() for c in u),
        "n_arroba": u.count("@"),
        "n_parametros": p.query.count("="),
        "profundidad_path": path.count("/"),
        "tiene_ip": int(es_ip),
        "usa_https": int(low.startswith("https")),
        "tiene_puerto": tiene_puerto,
        "tld_sospechoso": int(tld in TLD_SOSPECHOSOS),
        "es_acortador": int(host in ACORTADORES),
        "n_palabras_sospechosas": sum(w in low for w in PALABRAS_URL),
        "entropia": _entropia(host + path),
        "punycode": int("xn--" in host),
        "doble_slash_path": int("//" in path),
    }


def url_features_df(urls):
    return pd.DataFrame([url_features(u) for u in urls])


def text_indicators(texto):
    t = _norm(texto)
    out = {}
    for nombre, palabras in INDICADORES_TEXTO.items():
        hallados = [p for p in palabras if p in t]
        if hallados:
            out[nombre] = hallados
    return out


def url_indicators(url):
    f = url_features(url)
    razones = []
    if f["tiene_ip"]:
        razones.append("usa una dirección IP en lugar de un dominio")
    if not f["usa_https"]:
        razones.append("no usa HTTPS")
    if f["n_arroba"]:
        razones.append("contiene el símbolo @")
    if f["n_subdominios"] >= 3:
        razones.append("tiene muchos subdominios")
    if f["es_acortador"]:
        razones.append("es un acortador de enlaces")
    if f["tld_sospechoso"]:
        razones.append("usa una terminación de dominio poco habitual")
    if f["n_palabras_sospechosas"] >= 2:
        razones.append("contiene varias palabras típicas de fraude (login, verifica, banco...)")
    if f["longitud"] > 75:
        razones.append("es muy larga")
    if f["punycode"]:
        razones.append("usa caracteres camuflados (punycode)")
    if f["n_guiones_dominio"] >= 2:
        razones.append("el dominio tiene muchos guiones")
    return razones

class FusionPonderada:
    """Fusión sin aprendizaje: usa probabilidades [texto, url, tiene_url].
    La probabilidad no se recalibra al cambiar el umbral; aplicar .predict()
    o comparar predict_proba(... )[:, 1] con .umbral.
    """
    def __init__(self, peso_texto=1.0, umbral=0.5):
        self.peso_texto = float(peso_texto)
        self.umbral = float(umbral)

    def predict_proba(self, X):
        import numpy as np
        x = np.asarray(X, dtype=float)
        if x.ndim != 2 or x.shape[1] != 3:
            raise ValueError('Se necesitan 3 columnas: prob_texto, prob_url, tiene_url')
        pt = np.clip(x[:, 0], 0, 1)
        pu = np.clip(x[:, 1], 0, 1)
        tiene = x[:, 2] > 0.5
        p = np.where(tiene, self.peso_texto * pt +
                     (1 - self.peso_texto) * pu, pt)
        return np.column_stack((1 - p, p))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.umbral).astype(int)


In [ ]:
# 2. Librerías y lectura CSV
import sys, time, json, os, csv
from pathlib import Path
sys.path.insert(0, '/content')
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from features import extract_urls, url_features_df, FusionPonderada

def cargar(path):
    with open(path, 'r', encoding='utf-8-sig', errors='replace', newline='') as f:
        cabecera = f.readline()
    try:
        sep = csv.Sniffer().sniff(cabecera, delimiters=',;\t').delimiter
    except csv.Error:
        sep = ','
    try:
        datos = pd.read_csv(path, sep=sep, encoding='utf-8-sig', low_memory=False)
    except UnicodeDecodeError:
        datos = pd.read_csv(path, sep=sep, encoding='latin-1', low_memory=False)
    print(f'Archivo: {Path(path).name} | separador: {repr(sep)} | columnas: {len(datos.columns)}')
    return datos

print('Preparado: 1 = phishing; 0 = legítimo.')


## 3. Cargar y explorar los correos

In [ ]:
# 3. Cargar y explorar correos de SpaPhish
# La sección 2 debe estar ejecutada para disponer de cargar().
df = cargar(EMAIL_PATH)
print('Dimensiones:', df.shape)
print('Columnas:', df.columns.tolist())
display(df.head(3))

columnas_necesarias = {'subject', 'body', 'Label', 'urls'}
faltantes = columnas_necesarias - set(df.columns)
if faltantes:
    raise ValueError(f'Columnas necesarias ausentes: {sorted(faltantes)}. Revisa delimitador CSV.')


In [ ]:
# 3.1. Preparar los campos reales de SpaPhish
# Convención unificada: 1 = phishing, 0 = no phishing.
df['y'] = pd.to_numeric(df['Label'], errors='coerce')
df = df[df['y'].isin([0, 1])].copy()
df['y'] = df['y'].astype(int)

# Solo se usan texto y enlaces; el resto de atributos del dataset permanece intacto.
df['texto'] = (
    df['subject'].fillna('').astype(str).str.strip() + ' ' +
    df['body'].fillna('').astype(str).str.strip()
).str.strip()

# SpaPhish guarda la columna urls como una lista serializada en CSV.
# Extraemos enlaces tanto del texto como de ese campo; no accedemos a los sitios.
def urls_de_fila(fila):
    encontrados = extract_urls(fila['texto'])
    raw = fila.get('urls', '')
    if isinstance(raw, str):
        encontrados.extend(extract_urls(raw))
    return list(dict.fromkeys(encontrados))

df['urls'] = df.apply(urls_de_fila, axis=1)
df = df[df['texto'].str.len().gt(0)].copy()

# La división aleatoria puede poner el MISMO correo en train y test.
# Eliminar repeticiones exactas (normalizando solo espacios y mayúsculas).
# Si el mismo contenido tiene ambas etiquetas, descartarlo por inconsistente.
df['firma_texto'] = df['texto'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
contradicciones = df.groupby('firma_texto')['y'].nunique()
firmas_ambiguas = contradicciones[contradicciones > 1].index
n_conflictos = int(df['firma_texto'].isin(firmas_ambiguas).sum())
df = df[~df['firma_texto'].isin(firmas_ambiguas)].copy()
n_duplicados = int(df.duplicated('firma_texto').sum())
df = df.drop_duplicates('firma_texto').reset_index(drop=True)
print(f'Correos repetidos eliminados: {n_duplicados} | etiquetas contradictorias descartadas: {n_conflictos}')

print('Correos tras preparar:', len(df))
print('Clases: 1 = phishing, 0 = legítimo')
print(df['y'].value_counts().sort_index())
print('Correos con URL:', int(df['urls'].str.len().gt(0).sum()), 'de', len(df))
if df['y'].nunique() != 2:
    raise ValueError('Se requieren ambas clases (0 y 1) para entrenar.')

fig, ax = plt.subplots(figsize=(5, 3))
df['y'].value_counts().sort_index().rename({0:'Legítimo', 1:'Phishing'}).plot(kind='bar', ax=ax)
ax.set_title('SpaPhish: distribución de clases')
ax.set_ylabel('Número de correos')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()


## 4. División: 60 % train / 20 % validación / 20 % test
Se mantiene el porcentaje original. **Toda elección** de modelo, mezcla y umbral se basa exclusivamente en validación. El test se utiliza únicamente al final. La eliminación de duplicados exactos de texto ocurre **antes** de partir los datos, para reducir fuga de información.


In [ ]:
# 4. División estratificada 60/20/20
SEED = 42
y = df['y'].to_numpy()
idx = np.arange(len(df))
tr, tmp = train_test_split(idx, test_size=0.40, stratify=y, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=SEED)
print('Train:', len(tr), '| Validación:', len(va), '| Test:', len(te))

def metricas(y_true, p, umbral=0.5):
    y_true = np.asarray(y_true, dtype=int)
    p = np.asarray(p, dtype=float)
    yp = (p >= umbral).astype(int)
    return dict(
        Accuracy=accuracy_score(y_true, yp),
        Precision=precision_score(y_true, yp, zero_division=0),
        Recall=recall_score(y_true, yp, zero_division=0),
        F1=f1_score(y_true, yp, zero_division=0),
        ROC_AUC=roc_auc_score(y_true, p) if len(np.unique(y_true)) == 2 else np.nan
    )

# Evaluación en test (no usar para escoger modelo, pesos ni umbral)
resultados = []
def evaluar(nombre, y_true, probabilidades, umbral=0.5):
    r = dict(Modelo=nombre, N=len(y_true), Umbral=round(float(umbral), 4),
             **metricas(y_true, probabilidades, umbral))
    resultados.append(r)
    return r

# Objetivo: maximizar conjuntamente precision y accuracy, tratando de conservar
# recall>=95 %. Si no existen opciones factibles, la función lo indicará.
RECALL_MINIMO = 0.95
UMBRALES = np.linspace(0.05, 0.95, 181)

def ajustar_umbral(y_val, p_val, recall_min=RECALL_MINIMO):
    filas = []
    for t in UMBRALES:
        m = metricas(y_val, p_val, t)
        filas.append(dict(Umbral=float(t), **m))
    cand = pd.DataFrame(filas)
    elegibles = cand[cand['Recall'] >= recall_min]
    cumple = not elegibles.empty
    if not cumple:
        elegibles = cand.copy()
        # Sin umbral que satisfaga la meta: priorizar recall y después calidad.
        elegibles = elegibles[elegibles['Recall'] == elegibles['Recall'].max()]
    elegibles = elegibles.copy()
    elegibles['Puntaje'] = 0.5 * elegibles['Precision'] + 0.5 * elegibles['Accuracy']
    elegibles['Cercania_05'] = -(elegibles['Umbral'] - 0.5).abs()
    fila = elegibles.sort_values(['Puntaje', 'F1', 'Accuracy', 'Cercania_05'], ascending=False).iloc[0]
    return dict(umbral=float(fila['Umbral']), precision=float(fila['Precision']),
                accuracy=float(fila['Accuracy']), recall=float(fila['Recall']),
                f1=float(fila['F1']), puntaje=float(fila['Puntaje']),
                cumple_recall=bool(cumple))


## 5. Modelo A — Texto: línea base vs. palabras + caracteres (selección en validación)

In [ ]:
# 5. Modelos de texto candidatos. No usar el test para elegir parámetros.
def word_tfidf():
    return TfidfVectorizer(
        max_features=30000, ngram_range=(1, 2), min_df=2,
        strip_accents='unicode', sublinear_tf=True)

def modelo_mixto(C=1.0):
    return Pipeline([
        ('tfidf', FeatureUnion([
            ('palabras', word_tfidf()),
            ('caracteres', TfidfVectorizer(
                analyzer='char_wb', ngram_range=(3, 5), min_df=2,
                max_features=60000, strip_accents='unicode', sublinear_tf=True))
        ])),
        ('clasificador', LogisticRegression(
            C=C, max_iter=3000, class_weight='balanced', random_state=SEED))
    ])

# Línea base original; útil para ver si realmente mejoran los cambios.
modelo_base = make_pipeline(
    TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                    sublinear_tf=True, min_df=2),
    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)
)

candidatos_texto = {
    'Texto original': modelo_base,
    'Palabras + caracteres (C=0.5)': modelo_mixto(0.5),
    'Palabras + caracteres (C=2)': modelo_mixto(2.0),
    'Palabras + caracteres (C=6)': modelo_mixto(6.0),
}

seleccion_texto = []
for nombre, modelo in candidatos_texto.items():
    print('Entrenando:', nombre)
    modelo.fit(df['texto'].iloc[tr], y[tr])
    p_va = modelo.predict_proba(df['texto'].iloc[va])[:, 1]
    opt = ajustar_umbral(y[va], p_va)
    seleccion_texto.append(dict(Modelo=nombre, AUC_val=roc_auc_score(y[va], p_va), **opt))

ranking_texto = pd.DataFrame(seleccion_texto).sort_values(
    ['cumple_recall', 'puntaje', 'AUC_val'], ascending=False
)
display(ranking_texto.round(4))

nombre_texto_optimo = ranking_texto.iloc[0]['Modelo']
modelo_texto = candidatos_texto[nombre_texto_optimo]
print('Mejor configuración textual según VALIDACIÓN:', nombre_texto_optimo)

# Las probabilidades del test aún NO se evalúan ni se usan para elegir.
p_texto_va = modelo_texto.predict_proba(df['texto'].iloc[va])[:, 1]
p_texto_te = modelo_texto.predict_proba(df['texto'].iloc[te])[:, 1]
p_base_te = modelo_base.predict_proba(df['texto'].iloc[te])[:, 1]


## 6. Modelo B — URL (Random Forest)
LegitPhish tiene **0 = phishing, 1 = legítimo**. Se invierte su columna `ClassLabel` para usar el criterio común **1 = phishing, 0 = legítimo**. Se mantiene la división independiente de URLs 80/20, como en el original. La evaluación del test de URL es distinta del test de correos.


In [ ]:
# 6. Cargar LegitPhish y entrenar Random Forest
ext = cargar(URL_PATH)
print('Dimensiones LegitPhish:', ext.shape)
print('Columnas:', ext.columns.tolist())

# Elegir columnas por nombre exacto, no por coincidencia parcial (url_length NO es URL).
colmap = {str(c).strip().lower(): c for c in ext.columns}
cu = colmap.get('url')
cl = colmap.get('classlabel')
if cu is None or cl is None:
    raise ValueError(f'Se esperaban URL y ClassLabel; encontradas: {list(ext.columns)}')

etiqueta_original = pd.to_numeric(ext[cl], errors='coerce')
if not etiqueta_original.dropna().isin([0, 1]).all():
    raise ValueError('ClassLabel contiene valores fuera de {0, 1}. Revisa tu CSV.')

# LegitPhish: 0 = phishing, 1 = legítimo. Invertir al estándar común:
url_df = pd.DataFrame({
    'url': ext[cu].astype('string').str.strip(),
    'y': 1 - etiqueta_original
}).dropna(subset=['url','y'])
url_df = url_df[url_df['url'].str.len().gt(0)].copy()
url_df['y'] = url_df['y'].astype(int)

# Eliminar las URLs contradictorias y repetidas para evitar fugas entre train y test.
conflictos = url_df.groupby('url')['y'].nunique()
conflictos = conflictos[conflictos > 1].index
if len(conflictos):
    url_df = url_df[~url_df['url'].isin(conflictos)]
url_df = url_df.drop_duplicates(subset=['url']).reset_index(drop=True)
print('URLs únicas:', len(url_df), '| eliminadas por conflicto:', len(conflictos))
print('Etiquetas unificadas:', url_df['y'].value_counts().sort_index().to_dict())
if url_df['y'].nunique() != 2:
    raise ValueError('No hay dos clases de URL válidas en LegitPhish.')

# La evaluación de URL se hace solo en URLs nunca utilizadas para entrenarla.
url_tr, url_te = train_test_split(
    url_df, test_size=0.20, stratify=url_df['y'], random_state=42
)
# Limitar entrenamiento para mantener tiempos razonables en Google Colab.
if len(url_tr) > 60000:
    url_tr, _ = train_test_split(
        url_tr, train_size=60000, stratify=url_tr['y'], random_state=42
    )
print('URL train:', len(url_tr), '| URL test:', len(url_te))

Xu_tr = url_features_df(url_tr['url'].tolist())
Xu_te = url_features_df(url_te['url'].tolist())
modelo_url = RandomForestClassifier(
    n_estimators=200, n_jobs=-1, class_weight='balanced', random_state=42
)
modelo_url.fit(Xu_tr, url_tr['y'])
pu = modelo_url.predict_proba(Xu_te)[:, 1]
print('Evaluación URL (holdout LegitPhish):')
print(evaluar('Solo URL (dataset URL)', url_te['y'].values, pu))

imp = pd.Series(modelo_url.feature_importances_, index=Xu_tr.columns).sort_values()
imp.plot(kind='barh', figsize=(7, 6), title='Importancia de características de URL')
plt.tight_layout(); plt.show()


## 7. Probabilidad de URL por correo (máximo entre sus URLs)

In [ ]:
# 7. Combinar puntuaciones URL por correo: máximo entre sus URLs
def prob_url_correos(lista_urls):
    out = np.zeros(len(lista_urls), dtype=float)
    tiene = np.zeros(len(lista_urls), dtype=int)
    pares = [(i, u) for i, urls in enumerate(lista_urls) for u in urls]
    if pares:
        p = modelo_url.predict_proba(url_features_df([u for _, u in pares]))[:, 1]
        for (i, _), pp in zip(pares, p):
            out[i] = max(out[i], pp)
            tiene[i] = 1
    return out, tiene

p_url_va, has_va = prob_url_correos(df['urls'].iloc[va].tolist())
p_url_te, has_te = prob_url_correos(df['urls'].iloc[te].tolist())
print('Correos de validación con URL:', int(has_va.sum()), 'de', len(va))
print('Correos de test con URL:', int(has_te.sum()), 'de', len(te))


## 8. Mezcla optimizada de texto + URLs y umbral, seleccionados solo en validación

In [ ]:
# 8. Buscar pesos y umbrales usando SOLO correos de validación.
# Dar peso 1.0 equivale a ignorar URLs. Sin URL siempre usar solo texto.
PESOS_TEXTO = [0.50, 0.65, 0.80, 0.90, 0.95, 1.00]

def mezclar(p_texto, p_url, tiene, peso):
    return np.where(np.asarray(tiene) == 1,
                    peso * np.asarray(p_texto) + (1-peso) * np.asarray(p_url),
                    np.asarray(p_texto))

ranking_fusiones = []
for peso in PESOS_TEXTO:
    p_va_mix = mezclar(p_texto_va, p_url_va, has_va, peso)
    opt = ajustar_umbral(y[va], p_va_mix)
    ranking_fusiones.append(dict(peso_texto=peso, AUC_val=roc_auc_score(y[va], p_va_mix), **opt))

tabla_fusiones_val = pd.DataFrame(ranking_fusiones).sort_values(
    ['cumple_recall', 'puntaje', 'AUC_val', 'peso_texto'], ascending=False)
display(tabla_fusiones_val.round(4))

mejor = tabla_fusiones_val.iloc[0]
peso_optimo = float(mejor['peso_texto'])
umbral_optimo = float(mejor['umbral'])
fusion = FusionPonderada(peso_texto=peso_optimo, umbral=umbral_optimo)

p_fus_va = mezclar(p_texto_va, p_url_va, has_va, peso_optimo)
p_fus_te = mezclar(p_texto_te, p_url_te, has_te, peso_optimo)
p_prom_te = mezclar(p_texto_te, p_url_te, has_te, 0.5)

print(f'Configuración elegida en validación: peso texto={peso_optimo:.2f}, '
      f'peso URL={1-peso_optimo:.2f}, umbral={umbral_optimo:.3f}')
if not bool(mejor['cumple_recall']):
    print('AVISO: ningún umbral de los evaluados cumplió recall >=', RECALL_MINIMO)
print('Resultado de validación (no es el resultado final):',
      metricas(y[va], p_fus_va, umbral_optimo))


## 9. Comparación FINAL en correos de test (sin seleccionar por estos resultados)

In [ ]:
# 9. Comparamos sobre el MISMO test de correos, sin optimizar contra él.
# También mostramos la línea base original y promedio 50/50 a umbral 0.5.
resultados = []
print(evaluar('Texto original (umbral 0.50)', y[te], p_base_te, 0.5))
print(evaluar('Texto mejorado (umbral 0.50)', y[te], p_texto_te, 0.5))
print(evaluar('Promedio texto/URL 50-50', y[te], p_prom_te, 0.5))
print(evaluar('Optimizado en validación', y[te], p_fus_te, umbral_optimo))

tabla = pd.DataFrame(resultados).round(4)
display(tabla)
tabla.to_csv('/content/models/resultados.csv', index=False)

ax = tabla.set_index('Modelo')[['Accuracy', 'Precision', 'Recall', 'F1']].plot(
    kind='bar', figsize=(12,5), ylim=(0,1.05),
    title='Evaluación final: mismo test para los 4 modelos')
plt.xticks(rotation=12, ha='right'); plt.tight_layout(); plt.show()
print('Atención: una precisión mayor no implica necesariamente mayor recall.')


In [ ]:
# 9.1. Matrices de confusión y curvas ROC. La última usa el umbral validado.
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
comparar = [
    ('Texto original', p_base_te, 0.5),
    ('Promedio 50/50', p_prom_te, 0.5),
    ('Optimizado', p_fus_te, umbral_optimo)
]
for ax, (nom, p, t) in zip(axs, comparar):
    ConfusionMatrixDisplay(
        confusion_matrix(y[te], (p >= t).astype(int), labels=[0,1]),
        display_labels=['Legítimo (0)', 'Phishing (1)']
    ).plot(ax=ax, colorbar=False)
    ax.set_title(f'{nom} · umbral {t:.2f}')
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 5))
for nom, p, t in comparar:
    fpr, tpr, _ = roc_curve(y[te], p)
    plt.plot(fpr, tpr, label=f'{nom} (AUC={roc_auc_score(y[te], p):.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Falsos positivos'); plt.ylabel('Verdaderos positivos')
plt.title('Curvas ROC del test reservado'); plt.legend()
plt.tight_layout(); plt.show()

from sklearn.metrics import classification_report
print('Reporte de clasificación — modelo optimizado (test):')
print(classification_report(y[te], (p_fus_te >= umbral_optimo).astype(int),
                            target_names=['No phishing (0)', 'Phishing (1)'],
                            zero_division=0))


## 10. Latencia y exportación (se guarda el umbral y el peso elegidos)

In [ ]:
# 10. Guardar una fusión reproducible y su umbral de decisión.
def analizar(texto):
    urls = extract_urls(texto)
    pt = float(modelo_texto.predict_proba([texto])[0,1])
    pu = float(modelo_url.predict_proba(url_features_df(urls))[:,1].max()) if urls else 0.0
    return float(fusion.predict_proba([[pt, pu, float(bool(urls))]])[0,1])

muestra = df['texto'].iloc[te].tolist()[:100]
t0 = time.perf_counter()
for t in muestra:
    analizar(t)
lat_ms = (time.perf_counter()-t0) / max(len(muestra), 1) * 1000
print(f'Latencia media por correo: {lat_ms:.1f} ms')

joblib.dump(modelo_texto, '/content/models/modelo_texto.pkl')
joblib.dump(modelo_url, '/content/models/modelo_url.pkl')
joblib.dump(fusion, '/content/models/modelo_fusion.pkl')

tam = {nombre: round(os.path.getsize(f'/content/models/{nombre}.pkl')/1e6, 2)
       for nombre in ['modelo_texto', 'modelo_url', 'modelo_fusion']}
metadata = {
    'latencia_ms_por_correo': round(lat_ms, 2), 'tamano_mb': tam,
    'n_train': int(len(tr)), 'n_val': int(len(va)), 'n_test': int(len(te)),
    'mapeo_clases': {'0': 'no phishing', '1': 'phishing'},
    'texto_seleccionado': nombre_texto_optimo,
    'peso_texto': peso_optimo, 'peso_url': 1 - peso_optimo,
    'umbral_decision': umbral_optimo,
    'recall_minimo_val': RECALL_MINIMO,
    'email_dataset': Path(EMAIL_PATH).name,
    'url_dataset': Path(URL_PATH).name,
    'nota': 'En la aplicación comparar la probabilidad con umbral_decision; no usar siempre 0.50.'
}
with open('/content/models/metricas_operativas.json','w',encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('Tamaño (MB):', tam)
print('Predicción en despliegue: phishing = (probabilidad >=',round(umbral_optimo,3),')')


## 11. Guardar y descargar los modelos

In [ ]:
# 11. Exportar modelos, resultados y funciones necesarias para la app.
import shutil
from google.colab import files

shutil.copy('/content/features.py', '/content/models/features.py')
for archivo in Path('/content/models').iterdir():
    if archivo.is_file():
        shutil.copy2(archivo, Path(BASE) / 'models' / archivo.name)

zip_ruta = shutil.make_archive('/content/models_phishing', 'zip', '/content/models')
print('Archivos guardados en Drive:', sorted(p.name for p in (Path(BASE) / 'models').iterdir()))
files.download(zip_ruta)
print('Listo. Descomprime el ZIP en la carpeta models/ de la aplicación Streamlit.')
